In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from skimpy.utils.tabdict import TabDict
import os
import configparser
from pytfa.io.json import load_json_model
from skimpy.io.yaml import load_yaml_model
from skimpy.analysis.oracle.minimum_fluxes import MinFLuxVariable
import numpy as np
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../utils")))
from joblib import Parallel, delayed
import pickle

UPPER_IX = 1000
PHYSIOLOGY = 'WT'

# Read configuration from config.ini (single instance, robust path)
config = configparser.ConfigParser()
config_path = '/kincancer/scripts/src/config.ini'
config.read(config_path)

# Path to data and model from config, using base_dir
base_dir = config['paths']['base_dir']

path_to_kmodel = os.path.abspath(os.path.join(base_dir, config['paths'][f'path_to_kmodel_{PHYSIOLOGY}']))
path_to_tmodel = os.path.abspath(os.path.join(base_dir, config['paths'][f'path_to_tmodel_{PHYSIOLOGY}']))
path_to_samples = os.path.abspath(os.path.join(base_dir, config['paths'][f'path_to_samples_{PHYSIOLOGY}']))
path_to_fcc = os.path.abspath(os.path.join(base_dir, config['paths'][f'path_to_fcc_{PHYSIOLOGY}']))
path_to_fcc_df_WT = os.path.abspath(os.path.join(base_dir, config['paths'][f'path_to_cc_df_WT']))
path_to_fcc_df_MUT = os.path.abspath(os.path.join(base_dir, config['paths'][f'path_to_cc_df_MUT']))

In [ ]:
# Load pytfa model
print('Loading TFA model from:', path_to_tmodel)
tmodel = load_json_model(path_to_tmodel)
samples = pd.read_csv(path_to_samples, index_col=0)

# Find the producing reactions for each BBB
bbb_producing_reactions = {}
for bbb in tmodel.reactions.biomass.reactants:
    if bbb.id.endswith('_n'):
        print(f'Converting {bbb.id} to cytosolic metabolite.')
        bbb = tmodel.metabolites.get_by_id(bbb.id[:-2] + '_c')  # Convert to cytosolic metabolite if necessary
    # Find the producing reactions of the BBB
    if bbb.id in ['h2o_c']:
        continue
    reactions = [r.id for r in bbb.reactions if r.lower_bound*r.metabolites[bbb] > 0]
    bbb_producing_reactions[bbb.id] = reactions

In [ ]:
def get_fccs_for_reactions(ss, rxns, path = path_to_fcc):
    '''Get FCCs for all provided reactions at one steady state'''
    from collections import defaultdict
    
    rxn_to_dfs = defaultdict(list)

    for i in range(0, 100, 10):
        try:
            with open(path.format(ss, i, i + 9), 'rb') as f:
                fcc = pickle.load(f)
            for rxn in rxns:
                try:
                    slice_df = fcc.slice_by('flux', rxn)
                    slice_df.columns = [f'{ss},' + str(col) for col in slice_df.columns]
                    rxn_to_dfs[rxn].append(slice_df)
                except KeyError:
                    continue  # This reaction may not be present
        except (FileNotFoundError, OSError):
            continue

    # Concatenate all slices per reaction
    return {
        rxn: pd.concat(dfs, axis=1) if dfs else pd.DataFrame()
        for rxn, dfs in rxn_to_dfs.items()
    }

def parallel_get_fccs(max_ss=600, rxns=['biomass'], n_jobs=100):
    from collections import defaultdict
    
    all_results = Parallel(n_jobs=n_jobs)(
        delayed(get_fccs_for_reactions)(i, rxns) for i in tqdm(range(max_ss), desc='Loading FCCs')
    )
    
    # Combine results for each reaction
    rxn_to_dfs = defaultdict(list)
    for result in all_results:
        for rxn, df in result.items():
            if not df.empty:
                rxn_to_dfs[rxn].append(df)

    return {
        rxn: pd.concat(dfs, axis=1) if dfs else pd.DataFrame()
        for rxn, dfs in rxn_to_dfs.items()
    }

def process_reactions(met, rxn_ids, max_ss=700, remove_outliers=True, path_to_save = path_to_fcc_df_WT):
    rxn_ids = list(rxn_ids) 
    name_for_saving = met+'_producing_reactions'
    if os.path.exists(path_to_save.format(name_for_saving)):
        print(f"Skipping {met} as it already exists.")
        return

    try:
        all_fccs = parallel_get_fccs(max_ss=max_ss, rxns=rxn_ids)
        print('Finished loading FCCs')
        weights = []
        fccs = []
        for rxn, df in all_fccs.items():
            print(rxn)
            if df.empty:
                print(f"No FCC data found for {rxn}")
                continue
            # Remove outliers
            if remove_outliers:
                df = remove_outliers_parallel(df, n_jobs=100)
            fccs.append(df.reindex(sorted(df.columns), axis=1))

            weights.append(samples.loc[:, tmodel.reactions.get_by_id(rxn).id] - samples.loc[:, tmodel.reactions.get_by_id(rxn).reverse_id])

        # Stack all fccs into a 3D numpy array: shape (n_samples, n_rows, n_cols)
        fcc_array = np.stack([fcc.values for fcc in fccs])  # shape: (n_samples, n_rows, n_cols)

        # Extract column names and index from the first dataframe
        columns = fccs[0].columns
        index = fccs[0].index
        n_rows, n_cols = len(index), len(columns)

        # Precompute weights for each column based on 'ss'
        final_data = []

        for col_idx, col_name in enumerate(columns):
            ss = int(col_name.split(',')[0])
            col_values = fcc_array[:, :, col_idx]  # shape: (n_samples, n_rows)
            
            # Extract weights for this ss
            w = np.array([weight[ss] for weight in weights])  # shape: (n_samples,)
            
            numerator = np.tensordot(w, col_values, axes=(0, 0))  # shape: (n_rows,)
            denominator = w.sum()
            
            if denominator != 0:
                final_data.append(numerator / denominator)
            else:
                final_data.append(np.zeros(n_rows))

        # Stack column-wise and convert to DataFrame
        final_fcc = pd.DataFrame(np.column_stack(final_data), columns=columns, index=index)

        # Save the final FCCs for this metabolite
        final_fcc.to_csv(path_to_save.format(name_for_saving))
        print(f"Finished processing {met} with {len(rxn_ids)} reactions.")

        # Delete the fccs list to free memory
        del fccs
        del fcc_array
        del final_data

    except Exception as e:
        print(f"Failed for reactions {rxn_ids}: {e}")


In [ ]:
average_fccs_WT = pd.DataFrame(columns=bbb_producing_reactions.keys())
from tqdm import tqdm
for met, rxns in bbb_producing_reactions.items():
    name_for_saving = 'synthesis_rate_' + met
    filename = path_to_fcc_df_WT.format(name_for_saving)

    # First, get column names to define dtypes
    total_flux_df_cols = pd.read_csv(filename, index_col=0, nrows=0).columns
    dtype_dict = {col: 'float32' for col in total_flux_df_cols}
    dtype_dict['model_ix'] = 'string'

    # Count total number of lines for progress bar
    total_lines = sum(1 for _ in open(filename)) - 1  # Subtract header
    chunksize = 1000

    # Read in chunks using index_col=0 to preserve the original index
    chunk_iter = pd.read_csv(filename, chunksize=chunksize, index_col=0, dtype=dtype_dict)

    df_chunks = []
    for chunk in tqdm(chunk_iter, total=total_lines // chunksize + 1, desc="Loading CSV"):
        df_chunks.append(chunk)

    # Concatenate chunks WITHOUT ignore_index so original index is preserved
    total_flux_df = pd.concat(df_chunks)

    # Keep the average FCCs
    average_fccs_WT[met] = total_flux_df.mean(axis=1)

In [ ]:
average_fccs_MUT = pd.DataFrame(columns=bbb_producing_reactions.keys())
from tqdm import tqdm
for met, rxns in bbb_producing_reactions.items():
    name_for_saving = 'synthesis_rate_' + met
    filename = path_to_fcc_df_MUT.format(name_for_saving)

    # First, get column names to define dtypes
    total_flux_df_cols = pd.read_csv(filename, index_col=0, nrows=0).columns
    dtype_dict = {col: 'float32' for col in total_flux_df_cols}
    dtype_dict['model_ix'] = 'string'

    # Count total number of lines for progress bar
    total_lines = sum(1 for _ in open(filename)) - 1  # Subtract header
    chunksize = 1000

    # Read in chunks using index_col=0 to preserve the original index
    chunk_iter = pd.read_csv(filename, chunksize=chunksize, index_col=0, dtype=dtype_dict)

    df_chunks = []
    for chunk in tqdm(chunk_iter, total=total_lines // chunksize + 1, desc="Loading CSV"):
        df_chunks.append(chunk)

    # Concatenate chunks WITHOUT ignore_index so original index is preserved
    total_flux_df = pd.concat(df_chunks)

    # Keep the average FCCs
    average_fccs_MUT[met] = total_flux_df.mean(axis=1)

In [ ]:
# Remove the columns of essential amino acids and non essential amino acids
group_df = pd.DataFrame([
    ("his_L_c", "Amino Acid", "Essential"),
    ("ile_L_c", "Amino Acid", "Essential"),
    ("leu_L_c", "Amino Acid", "Essential"),
    ("lys_L_c", "Amino Acid", "Essential"),
    ("met_L_c", "Amino Acid", "Essential"),
    ("phe_L_c", "Amino Acid", "Essential"),
    ("thr_L_c", "Amino Acid", "Essential"),
    ("trp_L_c", "Amino Acid", "Essential"),
    ("val_L_c", "Amino Acid", "Essential"),
    ("ala_L_c", "Amino Acid", "Non-Essential"),
    ("arg_L_c", "Amino Acid", "Non-Essential"),
    ("asn_L_c", "Amino Acid", "Non-Essential"),
    ("asp_L_c", "Amino Acid", "Non-Essential"),
    ("cys_L_c", "Amino Acid", "Non-Essential"),
    ("gln_L_c", "Amino Acid", "Non-Essential"),
    ("glu_L_c", "Amino Acid", "Non-Essential"),
    ("gly_c", "Amino Acid", "Non-Essential"),
    ("pro_L_c", "Amino Acid", "Non-Essential"),
    ("ser_L_c", "Amino Acid", "Non-Essential"),
    ("tyr_L_c", "Amino Acid", "Non-Essential"),
    ("atp_c", "Ribonucleotide", "Nucleotides"),
    ("ctp_c", "Ribonucleotide", "Nucleotides"),
    ("gtp_c", "Ribonucleotide", "Nucleotides"),
    ("utp_c", "Ribonucleotide", "Nucleotides"),
    ("datp_c", "Deoxyribonucleotide", "Deoxynucleotides"),
    ("dctp_c", "Deoxyribonucleotide", "Deoxynucleotides"),
    ("dgtp_c", "Deoxyribonucleotide", "Deoxynucleotides"),
    ("dttp_c", "Deoxyribonucleotide", "Deoxynucleotides"),
    ("chsterol_c", "Lipid", "Lipids"),
    ("clpn_hs_c", "Lipid", "Lipids"),
    ("pail_hs_c", "Lipid", "Lipids"),
    ("pchol_hs_c", "Lipid", "Lipids"),
    ("pe_hs_c", "Lipid", "Lipids"),
    ("pglyc_hs_c", "Lipid", "Lipids"),
    ("ps_hs_c", "Lipid", "Lipids"),
    ("sphmyln_hs_c", "Lipid", "Lipids"),
    ("g6p_c", "Carbohydrate", "")
], columns=["Metabolite", "Category", "Subcategory"])

average_fccs_MUT = average_fccs_MUT.drop(columns=group_df[group_df['Subcategory'] == 'Essential']['Metabolite'].tolist())
average_fccs_MUT = average_fccs_MUT.drop(columns=group_df[group_df['Subcategory'] == 'Non-Essential']['Metabolite'].tolist())

average_fccs_WT = average_fccs_WT.drop(columns=group_df[group_df['Subcategory'] == 'Essential']['Metabolite'].tolist())
average_fccs_WT = average_fccs_WT.drop(columns=group_df[group_df['Subcategory'] == 'Non-Essential']['Metabolite'].tolist())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import OrderedDict

# Subset and clean index for average_fccs
mca_targets = [
 'vmax_forward_TRIOK', 'vmax_forward_MI1PP', 'vmax_forward_PPAP', 'vmax_forward_r0301', 'vmax_forward_METAT',
 'vmax_forward_3DSPHR', 'vmax_forward_TMDS', 'vmax_forward_SERPT', 'vmax_forward_HMR_7748', 'vmax_forward_PSP_L',
 'vmax_forward_NADH2_u10mi', 'vmax_forward_DGK1', 'vmax_forward_NTD1', 'vmax_forward_r0354', 'vmax_forward_ADSS',
 'vmax_forward_r0178', 'vmax_forward_IMPD', 'vmax_forward_r0179', 'vmax_forward_PGI', 'vmax_forward_r0474',
 'vmax_forward_ICDHyrm', 'vmax_forward_GMPS2', 'vmax_forward_UMPK2', 'vmax_forward_ICDHxm', 'vmax_forward_URIK1',
 'vmax_forward_CYTK1', 'vmax_forward_HMR_4343', 'vmax_forward_r0426'
]


sorted_names = ['TRIOK', 'HMR_7748', 'r0354', 'PGI', 'ICDHyrm', 'ICDHxm', 'r0426', 'PPAP', 'r0301', '3DSPHR', 'SERPT',
                 'METAT', 'PSP_L', 'r0178', 'r0179', 'TMDS', 'NTD1', 'DGK1', 'ADSS', 'IMPD', 'r0474', 'URIK1', 'GMPS2',
                   'HMR_4343', 'CYTK1', 'UMPK2', 'MI1PP', 'NADH2_u10mi']

data_to_plot_MUT = average_fccs_MUT.loc[set(mca_targets)]
data_to_plot_MUT.index = [i.split('vmax_forward_',)[1] for i in data_to_plot_MUT.index]
data_to_plot_MUT = data_to_plot_MUT.loc[(sorted_names)]
# Replace NADH2_u10mi in the index with Mcomplex1
data_to_plot_MUT = data_to_plot_MUT.rename(index={"NADH2_u10mi": "Mcomplex1"})

data_to_plot_WT = average_fccs_WT.loc[set(mca_targets)]
data_to_plot_WT.index = [i.split('vmax_forward_',)[1] for i in data_to_plot_WT.index]
data_to_plot_WT = data_to_plot_WT.loc[(sorted_names)]
# Replace NADH2_u10mi in the index with Mcomplex1
data_to_plot_WT = data_to_plot_WT.rename(index={"NADH2_u10mi": "Mcomplex1"})

# Create the mapping from metabolite to subcategory
group_df = pd.DataFrame([
    ("his_L_c", "Amino Acid", "Essential"),
    ("ile_L_c", "Amino Acid", "Essential"),
    ("leu_L_c", "Amino Acid", "Essential"),
    ("lys_L_c", "Amino Acid", "Essential"),
    ("met_L_c", "Amino Acid", "Essential"),
    ("phe_L_c", "Amino Acid", "Essential"),
    ("thr_L_c", "Amino Acid", "Essential"),
    ("trp_L_c", "Amino Acid", "Essential"),
    ("val_L_c", "Amino Acid", "Essential"),
    ("ala_L_c", "Amino Acid", "Non-Essential"),
    ("arg_L_c", "Amino Acid", "Non-Essential"),
    ("asn_L_c", "Amino Acid", "Non-Essential"),
    ("asp_L_c", "Amino Acid", "Non-Essential"),
    ("cys_L_c", "Amino Acid", "Non-Essential"),
    ("gln_L_c", "Amino Acid", "Non-Essential"),
    ("glu_L_c", "Amino Acid", "Non-Essential"),
    ("gly_c", "Amino Acid", "Non-Essential"),
    ("pro_L_c", "Amino Acid", "Non-Essential"),
    ("ser_L_c", "Amino Acid", "Non-Essential"),
    ("tyr_L_c", "Amino Acid", "Non-Essential"),
    ("atp_c", "Ribonucleotide", "Nucleotides"),
    ("ctp_c", "Ribonucleotide", "Nucleotides"),
    ("gtp_c", "Ribonucleotide", "Nucleotides"),
    ("utp_c", "Ribonucleotide", "Nucleotides"),
    ("datp_c", "Deoxyribonucleotide", "Deoxynucleotides"),
    ("dctp_c", "Deoxyribonucleotide", "Deoxynucleotides"),
    ("dgtp_c", "Deoxyribonucleotide", "Deoxynucleotides"),
    ("dttp_c", "Deoxyribonucleotide", "Deoxynucleotides"),
    ("chsterol_c", "Lipid", "Lipids"),
    ("clpn_hs_c", "Lipid", "Lipids"),
    ("pail_hs_c", "Lipid", "Lipids"),
    ("pchol_hs_c", "Lipid", "Lipids"),
    ("pe_hs_c", "Lipid", "Lipids"),
    ("pglyc_hs_c", "Lipid", "Lipids"),
    ("ps_hs_c", "Lipid", "Lipids"),
    ("sphmyln_hs_c", "Lipid", "Lipids"),
    ("g6p_c", "Carbohydrate", "")
], columns=["Metabolite", "Category", "Subcategory"])

# Reorder columns by subcategory
met_to_subcat = group_df.set_index("Metabolite")["Subcategory"].to_dict()
existing_columns = [col for col in data_to_plot_WT.columns if col in met_to_subcat]

subcategory_order = [
    'Essential',
    'Deoxynucleotides',
    'Nucleotides',
    "Non-Essential",
    "Lipids",
    ""
]

sorted_columns = sorted(
    existing_columns,
    key=lambda x: (subcategory_order.index(met_to_subcat[x]), x)
)
data_to_plot_WT = data_to_plot_WT[sorted_columns]
data_to_plot_MUT = data_to_plot_MUT[sorted_columns]

# Prepare subcategory boundaries for bracket plotting
subcat_to_indices = OrderedDict()
for i, col in enumerate(data_to_plot_WT.columns):
    subcat = met_to_subcat[col]
    if subcat not in subcat_to_indices:
        subcat_to_indices[subcat] = [i, i]
    else:
        subcat_to_indices[subcat][1] = i
        
# Function to add brackets to an axis
def add_brackets(ax, subcat_to_indices, met_to_subcat):
    # Positions just left of the heatmap axes area
    bracket_x = -3
    text_x = -4
    for label, (start, end) in subcat_to_indices.items():
        if label == '':
            continue
        if label == 'Essential':
            label = 'Essential AAs'
        elif label == 'Non-Essential':
            label = 'Non-Essential AAs'
        # Vertical line spanning the subcategory
        ax.vlines(bracket_x, start + 0.1, end + 1 - 0.1, color='black', linewidth=1.5, clip_on=False)
        # Horizontal tips
        ax.hlines([start + 0.1, end + 1 - 0.1], bracket_x, bracket_x + 0.6, color='black', linewidth=1.5, clip_on=False)
        # Label centered over the span
        ax.text(text_x, (start + end + 1) / 2, label,
                ha='center', va='center', fontsize=20, rotation=90, clip_on=False)

# Compute horizontal separator positions between adjacent subcategories (dashed lines)
# This avoids hardcoding indices and adapts to whatever columns are present
separator_positions = []
last_end = None
ordered_blocks = list(subcat_to_indices.items())
for idx, (label, (start, end)) in enumerate(ordered_blocks):
    if label == '':
        continue
    if idx > 0:
        # Separator goes at the top edge of this block
        separator_positions.append(start)

# Plot settings
VMIN, VMAX = -2, 2  # keep the same dynamic range as the previous figure for comparability

plt.rcParams.update({'font.size': 22})

fig = plt.figure(figsize=(35, 30))
ax1 = plt.subplot2grid((2, 25), (0, 0), colspan=20)   # WT
ax2 = plt.subplot2grid((2, 25), (1, 0), colspan=20)  # MUT

# Top heatmap (WT)
im_wt = sns.heatmap(
    data_to_plot_WT.T,
    cmap='seismic',
    center=0,
    vmin=VMIN,
    vmax=VMAX,
    cbar=False,
    ax=ax1,
    square=True
)
ax1.set_xlabel('')
ax1.set_ylabel('')
ax1.tick_params(axis='both', which='major', labelsize=18)
ax1.tick_params(axis='x', bottom=False, labelbottom=False)
add_brackets(ax1, subcat_to_indices, met_to_subcat)

# Bottom heatmap (MUT)
im_mut = sns.heatmap(
    data_to_plot_MUT.T,
    cmap='seismic',
    center=0,
    vmin=VMIN,
    vmax=VMAX,
    cbar=False,
    ax=ax2,
    square=True
)
ax2.set_xlabel('Enzymes', fontsize=26)
ax2.set_ylabel('')
ax2.tick_params(axis='both', which='major', labelsize=18)
add_brackets(ax2, subcat_to_indices, met_to_subcat)

# Inset colorbar to the right of the MUT heatmap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
cax = inset_axes(
    ax2,
    width="2%",
    height="45%",
    loc='center left',
    bbox_to_anchor=(1.05, 0.35, 1, 1),
    bbox_transform=ax2.transAxes,
    borderpad=0
)
# Use the right heatmap's mappable for the colorbar
cbar = plt.colorbar(im_mut.collections[0], cax=cax)
cbar.set_label('Effect on each biomass precursor', fontsize=20, labelpad=6)
cbar.set_ticks([VMIN, 0, VMAX])
cbar.set_ticklabels([f'< {VMIN}', '0', f'> {VMAX}'])
cbar.ax.tick_params(labelsize=20, length=3, width=1)

plt.tight_layout()

plt.show()

## Fast cached inputs for Figure 2D

In [ ]:
import os
import configparser
import pandas as pd

config = configparser.ConfigParser()
for config_path in ('../src/config.ini', 'scripts/src/config.ini', '/kincancer/scripts/src/config.ini'):
    if os.path.exists(config_path):
        config.read(config_path)
        break
else:
    raise FileNotFoundError('Could not find scripts/src/config.ini')

base_dir = config['paths']['base_dir']
path_to_ccc_df_WT = os.path.abspath(os.path.join(base_dir, config['paths']['path_to_cc_df_WT']))
path_to_ccc_df_MUT = os.path.abspath(os.path.join(base_dir, config['paths']['path_to_cc_df_MUT']))

# Figure 2D excludes essential and non-essential amino acids, so only these matrices are needed.
met_ids = [
    'datp_n', 'dctp_n', 'dgtp_n', 'dttp_n',
    'atp_c', 'ctp_c', 'gtp_c', 'utp_c',
    'chsterol_c', 'clpn_hs_c', 'pail_hs_c', 'pchol_hs_c',
    'pe_hs_c', 'pglyc_hs_c', 'ps_hs_c', 'sphmyln_hs_c',
    'g6p_c',
]

figure_2d_cache_dir = os.path.abspath(os.path.join(base_dir, 'results/mca_workflow/figure_2D_cached_inputs'))
os.makedirs(figure_2d_cache_dir, exist_ok=True)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def load_average_cccs(physiology, path_template):
    cache_path = os.path.join(figure_2d_cache_dir, f'average_cccs_{physiology}_figure_2D.csv')
    if os.path.exists(cache_path):
        print(f'Loading cached Figure 2D averages: {cache_path}')
        return pd.read_csv(cache_path, index_col=0)

    missing = [met_id for met_id in met_ids if not os.path.exists(path_template.format(met_id))]
    if missing:
        raise FileNotFoundError(f'Missing {physiology} CC CSVs for: {missing}')

    def average_one_metabolite(met_id):
        global_ccc = pd.read_csv(path_template.format(met_id), index_col=0)
        average_series = global_ccc.mean(axis=1)
        average_series.name = met_id
        return average_series

    max_workers = min(8, len(met_ids))
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        averages = list(executor.map(average_one_metabolite, met_ids))

    average_df = pd.concat(averages, axis=1)
    average_df.to_csv(cache_path)
    print(f'Saved cached Figure 2D averages: {cache_path}')
    return average_df

# Variables expected by the Figure 2D plotting cell.
bbb_df_wt = load_average_cccs('WT', path_to_ccc_df_WT)
bbb_df = load_average_cccs('MUT', path_to_ccc_df_MUT)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

sorted_names = [
    'TRIOK', 'HMR_7748', 'r0354', 'PGI', 'ICDHyrm', 'ICDHxm',
    'r0426', 'PPAP', 'r0301', '3DSPHR', 'SERPT', 'METAT', 'PSP_L',
    'r0178', 'r0179', 'TMDS', 'NTD1', 'DGK1', 'ADSS', 'IMPD',
    'r0474', 'URIK1', 'GMPS2', 'HMR_4343', 'CYTK1', 'UMPK2',
    'MI1PP', 'NADH2_u10mi',
]
mca_targets = [f'vmax_forward_{name}' for name in sorted_names]

group_df = pd.DataFrame([
    ('datp_n', 'Deoxyribonucleotide', 'Deoxynucleotides'),
    ('dctp_n', 'Deoxyribonucleotide', 'Deoxynucleotides'),
    ('dgtp_n', 'Deoxyribonucleotide', 'Deoxynucleotides'),
    ('dttp_n', 'Deoxyribonucleotide', 'Deoxynucleotides'),
    ('atp_c', 'Ribonucleotide', 'Nucleotides'),
    ('ctp_c', 'Ribonucleotide', 'Nucleotides'),
    ('gtp_c', 'Ribonucleotide', 'Nucleotides'),
    ('utp_c', 'Ribonucleotide', 'Nucleotides'),
    ('chsterol_c', 'Lipid', 'Lipids'),
    ('clpn_hs_c', 'Lipid', 'Lipids'),
    ('pail_hs_c', 'Lipid', 'Lipids'),
    ('pchol_hs_c', 'Lipid', 'Lipids'),
    ('pe_hs_c', 'Lipid', 'Lipids'),
    ('pglyc_hs_c', 'Lipid', 'Lipids'),
    ('ps_hs_c', 'Lipid', 'Lipids'),
    ('sphmyln_hs_c', 'Lipid', 'Lipids'),
    ('g6p_c', 'Carbohydrate', ''),
], columns=['Metabolite', 'Category', 'Subcategory'])

met_to_subcat = group_df.set_index('Metabolite')['Subcategory'].to_dict()
subcategory_order = ['Deoxynucleotides', 'Nucleotides', 'Lipids', '']
sorted_columns = sorted(
    [col for col in met_ids if col in met_to_subcat],
    key=lambda col: (subcategory_order.index(met_to_subcat[col]), col),
)

data_to_plot_WT = bbb_df_wt.loc[mca_targets, sorted_columns].copy()
data_to_plot_MUT = bbb_df.loc[mca_targets, sorted_columns].copy()

for data_to_plot in (data_to_plot_WT, data_to_plot_MUT):
    data_to_plot.index = [idx.split('vmax_forward_')[1] for idx in data_to_plot.index]
    data_to_plot.rename(index={'NADH2_u10mi': 'Mcomplex1'}, inplace=True)

subcat_to_indices_y = OrderedDict()
for i, metabolite in enumerate(sorted_columns):
    subcat = met_to_subcat[metabolite]
    if subcat not in subcat_to_indices_y:
        subcat_to_indices_y[subcat] = [i, i]
    else:
        subcat_to_indices_y[subcat][1] = i

def add_brackets_y(ax, subcat_to_indices):
    bracket_x = -4.4
    text_x_by_label = {
        'Deoxynucleotides': -5.6,
        'Nucleotides': -5.0,
        'Lipids': -5.1,
    }
    for label, (start, end) in subcat_to_indices.items():
        if label == '':
            continue
        ax.vlines(bracket_x, start + 0.1, end + 1 - 0.1, color='black', linewidth=0.8, clip_on=False)
        ax.hlines([start + 0.1, end + 1 - 0.1], bracket_x, bracket_x + 0.5, color='black', linewidth=0.8, clip_on=False)
        text_x = text_x_by_label.get(label, -5.1)
        ax.text(text_x, (start + end + 1) / 2, label, ha='center', va='center', fontsize=8, rotation=90, clip_on=False)

display_columns = [metabolite[:-2] for metabolite in sorted_columns]
data_to_plot_WT.columns = display_columns
data_to_plot_MUT.columns = display_columns

VMIN, VMAX = -2, 2
plt.rcParams.update({'font.size': 22})
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(35, 20), gridspec_kw={'hspace': 0.05})

im_wt = sns.heatmap(
    data_to_plot_WT.T,
    cmap='seismic',
    center=0,
    vmin=VMIN,
    vmax=VMAX,
    cbar=False,
    ax=ax1,
    square=True,
)
ax1.set_ylabel('Precursors', fontsize=26, labelpad=50)
ax1.tick_params(axis='both', which='major', labelsize=18)
ax1.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
add_brackets_y(ax1, subcat_to_indices_y)

im_mut = sns.heatmap(
    data_to_plot_MUT.T,
    cmap='seismic',
    center=0,
    vmin=VMIN,
    vmax=VMAX,
    cbar=False,
    ax=ax2,
    square=True,
)
ax2.set_xlabel('Enzymes', fontsize=26)
ax2.set_ylabel('Precursors', fontsize=26, labelpad=50)
ax2.tick_params(axis='both', which='major', labelsize=18)
add_brackets_y(ax2, subcat_to_indices_y)

cax = inset_axes(
    ax2,
    width='2%',
    height='50%',
    loc='center left',
    bbox_to_anchor=(1.05, 0.35, 1, 1),
    bbox_transform=ax2.transAxes,
    borderpad=0,
)
cbar = plt.colorbar(im_mut.collections[0], cax=cax)
cbar.set_label('Effect on each biomass precursor', fontsize=20, labelpad=6)
cbar.set_ticks([VMIN, 0, VMAX])
cbar.set_ticklabels([f'< {VMIN}', '0', f'> {VMAX}'])
cbar.ax.tick_params(labelsize=20, length=3, width=1)

plt.show()

## Combined Figure 2C and Figure 2D

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict

required_inputs = ['average_fccs_WT', 'average_fccs_MUT', 'bbb_df_wt', 'bbb_df']
missing_inputs = [name for name in required_inputs if name not in globals()]
if missing_inputs:
    raise NameError(f'Run the Figure 2C and fast Figure 2D input cells first. Missing: {missing_inputs}')

sorted_names = [
    'TRIOK', 'HMR_7748', 'r0354', 'PGI', 'ICDHyrm', 'ICDHxm',
    'r0426', 'PPAP', 'r0301', '3DSPHR', 'SERPT', 'METAT', 'PSP_L',
    'r0178', 'r0179', 'TMDS', 'NTD1', 'DGK1', 'ADSS', 'IMPD',
    'r0474', 'URIK1', 'GMPS2', 'HMR_4343', 'CYTK1', 'UMPK2',
    'MI1PP', 'NADH2_u10mi',
]
mca_targets = [f'vmax_forward_{name}' for name in sorted_names]

figure_2c_met_ids = [
    'datp_c', 'dctp_c', 'dgtp_c', 'dttp_c',
    'atp_c', 'ctp_c', 'gtp_c', 'utp_c',
    'chsterol_c', 'clpn_hs_c', 'pail_hs_c', 'pchol_hs_c',
    'pe_hs_c', 'pglyc_hs_c', 'ps_hs_c', 'sphmyln_hs_c',
    'g6p_c',
]
figure_2d_met_ids = [
    'datp_n', 'dctp_n', 'dgtp_n', 'dttp_n',
    'atp_c', 'ctp_c', 'gtp_c', 'utp_c',
    'chsterol_c', 'clpn_hs_c', 'pail_hs_c', 'pchol_hs_c',
    'pe_hs_c', 'pglyc_hs_c', 'ps_hs_c', 'sphmyln_hs_c',
    'g6p_c',
]
display_met_ids = [met_id[:-2] for met_id in figure_2c_met_ids]

subcat_by_display_met = OrderedDict([
    ('datp', 'Deoxynucleotides'),
    ('dctp', 'Deoxynucleotides'),
    ('dgtp', 'Deoxynucleotides'),
    ('dttp', 'Deoxynucleotides'),
    ('atp', 'Nucleotides'),
    ('ctp', 'Nucleotides'),
    ('gtp', 'Nucleotides'),
    ('utp', 'Nucleotides'),
    ('chsterol', 'Lipids'),
    ('clpn_hs', 'Lipids'),
    ('pail_hs', 'Lipids'),
    ('pchol_hs', 'Lipids'),
    ('pe_hs', 'Lipids'),
    ('pglyc_hs', 'Lipids'),
    ('ps_hs', 'Lipids'),
    ('sphmyln_hs', 'Lipids'),
    ('g6p', ''),
])

def prepare_heatmap_data(source_df, metabolite_ids):
    data_to_plot = source_df.loc[mca_targets, metabolite_ids].copy()
    data_to_plot.index = [idx.split('vmax_forward_')[1] for idx in data_to_plot.index]
    data_to_plot.rename(index={'NADH2_u10mi': 'Mcomplex1'}, inplace=True)
    data_to_plot.columns = [met_id[:-2] for met_id in metabolite_ids]
    return data_to_plot

data_2c_wt = prepare_heatmap_data(average_fccs_WT, figure_2c_met_ids)
data_2c_mut = prepare_heatmap_data(average_fccs_MUT, figure_2c_met_ids)
data_2d_wt = prepare_heatmap_data(bbb_df_wt, figure_2d_met_ids)
data_2d_mut = prepare_heatmap_data(bbb_df, figure_2d_met_ids)

subcat_to_indices = OrderedDict()
for i, metabolite in enumerate(display_met_ids):
    subcat = subcat_by_display_met[metabolite]
    if subcat not in subcat_to_indices:
        subcat_to_indices[subcat] = [i, i]
    else:
        subcat_to_indices[subcat][1] = i

def add_brackets_y(ax, subcat_to_indices):
    bracket_x = -4.4
    text_x_by_label = {
        'Deoxynucleotides': -5.6,
        'Nucleotides': -5.0,
        'Lipids': -5.1,
    }
    for label, (start, end) in subcat_to_indices.items():
        if label == '':
            continue
        ax.vlines(bracket_x, start + 0.1, end + 1 - 0.1, color='black', linewidth=0.8, clip_on=False)
        ax.hlines([start + 0.1, end + 1 - 0.1], bracket_x, bracket_x + 0.5, color='black', linewidth=0.8, clip_on=False)
        text_x = text_x_by_label.get(label, -5.1)
        ax.text(text_x, (start + end + 1) / 2, label, ha='center', va='center', fontsize=8, rotation=90, clip_on=False)

VMIN, VMAX = -2, 2
plt.rcParams.update({'font.size': 10})
fig = plt.figure(figsize=(13.7, 8.7))
n_rows, n_cols = data_2c_wt.T.shape
axes_height = 0.36
axes_width = axes_height * (fig.get_figheight() / fig.get_figwidth()) * (n_cols / n_rows)
left = 0.11
column_gap = 0.095
bottom = 0.11
row_gap = 0.035
right = left + axes_width + column_gap
top = bottom + axes_height + row_gap

ax_2c_wt = fig.add_axes([left, top, axes_width, axes_height])
ax_2d_wt = fig.add_axes([right, top, axes_width, axes_height])
ax_2c_mut = fig.add_axes([left, bottom, axes_width, axes_height])
ax_2d_mut = fig.add_axes([right, bottom, axes_width, axes_height])

heatmap_kwargs = dict(cmap='seismic', center=0, vmin=VMIN, vmax=VMAX, cbar=False, square=True)
im_2c_wt = sns.heatmap(data_2c_wt.T, ax=ax_2c_wt, **heatmap_kwargs)
im_2d_wt = sns.heatmap(data_2d_wt.T, ax=ax_2d_wt, **heatmap_kwargs)
im_2c_mut = sns.heatmap(data_2c_mut.T, ax=ax_2c_mut, **heatmap_kwargs)
im_2d_mut = sns.heatmap(data_2d_mut.T, ax=ax_2d_mut, **heatmap_kwargs)

ax_2c_wt.set_title('Effect on the synthesis rate', fontsize=14, pad=6)
ax_2d_wt.set_title('Effect on the concentration', fontsize=14, pad=6)

for ax in (ax_2c_wt, ax_2d_wt):
    ax.set_xlabel('')
    ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)

for ax in (ax_2c_mut, ax_2d_mut):
    ax.set_xlabel('')

for ax in (ax_2c_wt, ax_2c_mut, ax_2d_wt, ax_2d_mut):
    ax.set_ylabel('')
    ax.tick_params(axis='both', which='major', labelsize=8)

for ax in (ax_2d_wt, ax_2d_mut):
    ax.tick_params(axis='y', left=False, labelleft=False)

ax_2c_wt.set_ylabel('WT', fontsize=16, labelpad=32, color='skyblue')
ax_2c_mut.set_ylabel('MUT', fontsize=16, labelpad=32, color='salmon')
add_brackets_y(ax_2c_wt, subcat_to_indices)
add_brackets_y(ax_2c_mut, subcat_to_indices)

cbar_item_width = 0.050
cbar_width = 0.012
cbar_height = 0.15
cbar_item_left = left + axes_width + (column_gap - cbar_item_width) / 2
cbar_left = cbar_item_left
cbar_bottom = 0.5 - cbar_height / 2
cax = fig.add_axes([cbar_left, cbar_bottom, cbar_width, cbar_height])
cbar = fig.colorbar(im_2d_mut.collections[0], cax=cax)
cbar.set_label('Control coefficient', fontsize=10, labelpad=4)
cbar.set_ticks([VMIN, 0, VMAX])
cbar.set_ticklabels([f'< {VMIN}', '0', f'> {VMAX}'])
cbar.ax.tick_params(labelsize=8, length=1.4, width=0.6)

output_dir = os.path.abspath(os.path.join(base_dir, 'results/figures'))
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'figure_2C_2D_combined.pdf')
fig.savefig(output_path, bbox_inches='tight')
print(f'Saved combined figure to: {output_path}')
plt.show()